# Feature Extraction in Text — Full Pipeline

This notebook runs the complete, hand-built pipeline end to end on real
documents scraped from Hacker News (`hackernews_dataset.csv`, produced by
`scrape_hackernews.py`), falling back to the small curated `demo.csv` corpus
if that file isn't present:

1. Load raw text from CSV
2. Preprocess (lowercase, strip punctuation/numbers, tokenize, remove
   stopwords, stem)
3. POS tagging
4. Syntactic features (POS tags -> parse tree -> subject/verb/object,
   noun/verb/adjective counts, clause and dependency information)
5. Semantic features (rule-based lexical categories)
6. Morphological analysis
7. Lemmatization (vs. stemming)
8. Vocabulary + Bag of Words / Binary BoW
9. One-hot encoding (tokens and POS tags)
10. TF-IDF + document cosine similarity
11. N-grams

All logic comes from `preprocessing.py`, `linguistic_features.py`,
`syntactic_features.py`, `semantic_features.py`, and `features.py` in this
project — no scikit-learn, nltk, or spaCy.

## 1. Setup

In [111]:
import pandas as pd

from preprocessing import (
    preprocess,
    lowercase,
    remove_punctuation_and_numbers,
    tokenize,
    remove_stopwords,
    stem_word,
    stem_tokens,
)
from linguistic_features import (
    pos_tag,
    pos_tag_word,
    morphological_analysis,
    lemmatize_word,
    lemmatize_tokens,
)
def build_parse_tree(tagged_tokens):
    """Build a lightweight constituency-style tree."""
    children = []

    for word, tag in tagged_tokens:
        if tag == "DET":
            label = "DET"
        elif tag in {"NOUN", "PRON"}:
            label = "NP"
        elif tag in {"VERB", "AUX"}:
            label = "VP"
        elif tag == "PREP":
            label = "PP"
        elif tag == "ADJ":
            label = "ADJP"
        else:
            label = tag

        children.append({
            "label": label,
            "word": word,
            "tag": tag,
            "children": [],
        })

    return {
        "label": "S",
        "word": None,
        "tag": None,
        "children": children,
    }


def render_tree(tree, indent=0):
    lines = []
    prefix = "  " * indent

    if tree["word"] is not None:
        lines.append(f"{prefix}{tree['label']}({tree['word']})")
    else:
        lines.append(f"{prefix}{tree['label']}")
        for child in tree["children"]:
            lines.extend(render_tree(child, indent + 1).splitlines())

    return "\n".join(lines)


def extract_dependencies(tree):
    dependencies = []
    words = tree["children"]

    nouns = [node for node in words if node["tag"] in {"NOUN", "PRON"}]
    verbs = [node for node in words if node["tag"] in {"VERB", "AUX"}]

    for node in words:
        if node["tag"] == "DET":
            following_noun = next(
                (n for n in nouns if words.index(n) > words.index(node)),
                None,
            )
            if following_noun:
                dependencies.append({
                    "relation": "det",
                    "head": following_noun["word"],
                    "dependent": node["word"],
                })

    if nouns and verbs:
        dependencies.append({
            "relation": "nsubj",
            "head": verbs[0]["word"],
            "dependent": nouns[0]["word"],
        })

    if len(nouns) > 1 and verbs:
        dependencies.append({
            "relation": "dobj",
            "head": verbs[0]["word"],
            "dependent": nouns[1]["word"],
        })

    return dependencies
from semantic_features import semantic_feature_matrix, cosine_similarity_matrix
from syntactic_features import syntactic_analysis, syntactic_feature_matrix
from features import (
    build_vocabulary,
    bag_of_words,
    binary_bow,
    tf_idf,
    n_grams,
    one_hot_encode_tokens,
    one_hot_encode_categories,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 2. Load raw text from CSV

Reads `hackernews_dataset.csv` (real scraped Hacker News stories, one per
row in a `text` column) if it exists, otherwise falls back to the small
curated `demo.csv` corpus.

In [112]:
from pathlib import Path

csv_path = "hackernews_dataset.csv" if Path("hackernews_dataset.csv").exists() else "demo.csv"
# csv_path = "demo.csv"
data = pd.read_csv(csv_path)
corpus = data["text"][:10].tolist()

print(f"Loaded {len(corpus)} documents from {csv_path}\n")
for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")

Loaded 10 documents from hackernews_dataset.csv

Doc 0: The Therapeutic Potential of Apigenin (2018)
Doc 1: Interactive Physics
Doc 2: The Economics of the Intelligence Frontier
Doc 3: Packslip – signed release manifest for archives, installers, or executables
Doc 4: The Innovative HP Computer Behind a Hedge Fund Pioneer
Doc 5: Federal judge again rules against Musk's xAI
Doc 6: TIL: Using Blender with coding agents on macOS
Doc 7: Show HN: MarkFlowy – A open source Markdown editor rebuilt for large documents. I maintain MarkFlowy, a desktop Markdown editor for Windows, macOS, and Linux. I’ve spent the past two months rebuilding its editor core, mainly to improve performance with larger documents. The changes are part of v0.100.0. In testing, a 2 MB Markdown document opened in about a second. I’d be interested to see how it performs with other people’s documents and hardware. Opening speed was one part of the work. I also worked on editing long documents and copying and cutting large s

## 3. Preprocessing

`preprocess()` runs lowercase -> strip punctuation/numbers -> tokenize ->
remove stopwords -> stem, in one call. We also keep a lighter "raw tokens"
version per document (lowercased and tokenized, but *not* stopword-stripped
or stemmed) for the linguistic analysis steps below, since POS tagging and
morphology need function words and full word forms to work with.


In [113]:
processed_docs = [preprocess(doc) for doc in corpus]
raw_tokens_per_doc = [
    tokenize(remove_punctuation_and_numbers(lowercase(doc))) for doc in corpus
]

print("=== Tokens after full preprocessing (stopwords removed, stemmed) ===")
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}: {tokens}")


=== Tokens after full preprocessing (stopwords removed, stemmed) ===
Doc 0: ['therapeutic', 'potential', 'apigenin']
Doc 1: ['interactive', 'physic']
Doc 2: ['economic', 'intelligence', 'frontier']
Doc 3: ['packslip', 'sign', 'release', 'manifest', 'archiv', 'installer', 'executabl']
Doc 4: ['innovative', 'hp', 'computer', 'behind', 'hedge', 'fund', 'pioneer']
Doc 5: ['federal', 'judge', 'rul', 'musk', 's', 'xai']
Doc 6: ['til', 'using', 'blender', 'cod', 'agent', 'maco']
Doc 7: ['show', 'hn', 'markflowy', 'open', 'source', 'markdown', 'editor', 'rebuilt', 'large', 'document', 'maintain', 'markflowy', 'desktop', 'markdown', 'editor', 'window', 'maco', 'linux', 've', 'spent', 'past', 'two', 'month', 'rebuild', 'editor', 'core', 'main', 'improve', 'performance', 'larger', 'document', 'chang', 'part', 'v', 'test', 'mb', 'markdown', 'document', 'open', 'second', 'd', 'interest', 'see', 'how', 'perform', 'other', 'people', 's', 'document', 'hardware', 'open', 'spe', 'one', 'part', 'work', '

## 4. POS Tagging

Rule-based tagging (closed-class lexicon + suffix rules) on the raw tokens
of every document, shown as one combined table.


In [114]:
pos_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    for word, tag in pos_tag(tokens):
        pos_rows.append({"doc": doc_index, "word": word, "pos_tag": tag})

pos_df = pd.DataFrame(pos_rows)
pos_df


,doc,word,pos_tag
0,0,the,DET
1,0,therapeutic,ADJ
2,0,potential,ADJ
3,0,of,PREP
4,0,apigenin,NOUN
...,...,...,...
256,9,ai,NOUN
257,9,video,NOUN
258,9,object,NOUN
259,9,remover,NOUN


## 5. Syntactic Features

Full pipeline: tokens -> POS tags -> a small hand-written constituency
parser -> a parse tree, from which we extract:

- **subject / verb / object** — read off the tree's first clause instead of
  guessed from raw token order
- **noun / verb / adjective counts**
- **clause count** — how many `S` (clause) nodes the parser found
- **dependency relations** (`det`, `amod`, `nsubj`, `dobj`, `pobj`) — simple
  head-dependent pairs read off the tree, approximating a dependency parse

In [128]:
print("=== Parse Tree (Doc 0) ===")
parse_tree_doc0 = build_parse_tree(pos_tag(raw_tokens_per_doc[8]))
print(render_tree(parse_tree_doc0))

print("\n=== Dependencies (Doc 0) ===")
for dependency in extract_dependencies(parse_tree_doc0):
    print(f"{dependency['relation']:6s} {dependency['head']} -> {dependency['dependent']}")

syntactic_frames = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    tagged = pos_tag(tokens)
    doc_syntax = syntactic_analysis(tagged).drop(
        columns=["dependencies"],
        errors="ignore",
    )
    doc_syntax.insert(0, "doc", doc_index)
    syntactic_frames.append(doc_syntax)

syntactic_df = pd.concat(syntactic_frames, ignore_index=True)
syntactic_df

=== Parse Tree (Doc 0) ===
S
  NP(how)
  NP(would)
  NP(llms)
  NP(vote)
  PP(in)
  VP(upcoming)
  NP(german)
  NP(state)
  NP(elections)

=== Dependencies (Doc 0) ===
nsubj  upcoming -> how
dobj   upcoming -> would


,doc,subject,verb,object,token_count,noun_count,verb_count,has_subject_verb_object
0,0,None,None,None,5,1,0,False
1,1,None,None,None,2,1,0,False
2,2,None,None,None,6,3,0,False
3,3,packslip,signed,release,9,6,1,True
4,4,None,None,None,9,6,0,False
5,5,None,None,None,8,6,0,False
6,6,til,using,blender,8,4,2,True
7,7,show,rebuilding,its,196,105,25,True
8,8,how,upcoming,german,9,7,1,True
9,9,None,None,None,9,9,0,False


## 6. Semantic Features

Counts of tokens per hand-written lexical category (action, animal, place,
object, positive_description, descriptive) from `SEMANTIC_LEXICON`.

In [116]:
semantic_df = semantic_feature_matrix(raw_tokens_per_doc)
semantic_df

,action,animal,place,object,positive_description,descriptive
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0
5,0,0,0,0,0,0
6,0,0,0,0,0,0
7,0,0,0,0,0,0
8,0,0,0,0,0,0
9,0,0,0,0,0,0


## 7. Morphological Analysis

Per-word shape and inflection features (length, vowel/consonant counts,
prefix/suffix, plural/gerund/past-tense/comparative/superlative flags) for
every document, combined into one table.

In [117]:
morph_frames = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    doc_morph = morphological_analysis(tokens)
    doc_morph.insert(0, "doc", doc_index)
    morph_frames.append(doc_morph)

morph_df = pd.concat(morph_frames, ignore_index=True)
morph_df


,doc,word,length,num_vowels,num_consonants,prefix3,suffix3,is_capitalized,is_plural,is_gerund,is_past_tense,is_comparative,is_superlative
0,0,the,3,1,2,the,the,False,False,False,False,False,False
1,0,therapeutic,11,5,6,the,tic,False,False,False,False,False,False
2,0,potential,9,4,5,pot,ial,False,False,False,False,False,False
3,0,of,2,1,1,of,of,False,False,False,False,False,False
4,0,apigenin,8,4,4,api,nin,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
256,9,ai,2,2,0,ai,ai,False,False,False,False,False,False
257,9,video,5,3,2,vid,deo,False,False,False,False,False,False
258,9,object,6,2,4,obj,ect,False,False,False,False,False,False
259,9,remover,7,3,4,rem,ver,False,False,False,False,True,False


## 8. Lemmatization vs. Stemming

Both reduce a word to a base form, but the stemmer just chops suffixes
(sometimes producing fragments that aren't real words), while the
lemmatizer aims to return an actual dictionary word. Comparison shown on
stopword-free tokens from every document.

In [118]:
lemma_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    filtered = remove_stopwords(tokens)
    for word in filtered:
        lemma_rows.append({
            "doc": doc_index,
            "word": word,
            "stem": stem_word(word),
            "lemma": lemmatize_word(word),
        })

lemma_df = pd.DataFrame(lemma_rows)
lemma_df


,doc,word,stem,lemma
0,0,therapeutic,therapeutic,therapeutic
1,0,potential,potential,potential
2,0,apigenin,apigenin,apigenin
3,1,interactive,interactive,interactive
4,1,physics,physic,physic
...,...,...,...,...
175,9,ai,ai,ai
176,9,video,video,video
177,9,object,object,object
178,9,remover,remover,remov


## 9. Vocabulary

Built from the fully preprocessed (stopword-free, stemmed) tokens.

In [119]:
vocab = build_vocabulary(processed_docs)
print(f"Vocabulary size: {len(vocab)}")
vocab


Vocabulary size: 141


['agent',
 'ai',
 'also',
 'anyth',
 'apigenin',
 'appreciate',
 'archiv',
 'awkward',
 'behavior',
 'behind',
 'blender',
 'bug',
 'came',
 'chang',
 'cod',
 'com',
 'computer',
 'copy',
 'core',
 'cutt',
 'd',
 'desktop',
 'document',
 'download',
 'drl',
 'economic',
 'edg',
 'edit',
 'editor',
 'election',
 'entire',
 'executabl',
 'federal',
 'feedback',
 'feel',
 'find',
 'focu',
 'frontier',
 'fund',
 'german',
 'get',
 'github',
 'hardware',
 'hedge',
 'help',
 'hn',
 'how',
 'hp',
 'http',
 'improve',
 'innovative',
 'installer',
 'intelligence',
 'interaction',
 'interactive',
 'interest',
 'judge',
 'large',
 'larger',
 'linux',
 'llm',
 'long',
 'maco',
 'main',
 'maintain',
 'manifest',
 'markdown',
 'markflowy',
 'mb',
 'month',
 'much',
 'musk',
 'object',
 'one',
 'online',
 'open',
 'operation',
 'other',
 'overhead',
 'packslip',
 'part',
 'past',
 'people',
 'perform',
 'performance',
 'physic',
 'pioneer',
 'potential',
 'process',
 'project',
 'rebuild',
 'rebuilt'

## 10. Bag of Words

In [120]:
bow_df = bag_of_words(processed_docs, vocab)
bow_df


,agent,ai,also,anyth,apigenin,appreciate,archiv,awkward,behavior,behind,blender,bug,came,chang,cod,...,user,using,v,ve,video,vote,want,way,where,who,window,within,work,would,xai
0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
6,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,1,1,0,1,0,1,1,0,0,1,1,2,0,...,1,0,1,1,0,0,1,1,1,1,1,1,6,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
9,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


## 11. Binary Bag of Words

In [121]:
binary_df = binary_bow(processed_docs, vocab)
binary_df


,agent,ai,also,anyth,apigenin,appreciate,archiv,awkward,behavior,behind,blender,bug,came,chang,cod,...,user,using,v,ve,video,vote,want,way,where,who,window,within,work,would,xai
0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
6,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,1,1,0,1,0,1,1,0,0,1,1,1,0,...,1,0,1,1,0,0,1,1,1,1,1,1,1,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
9,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


## 12. One-Hot Encoding

Two flavors: one-hot per token *position* in a document (preserves order,
unlike bag-of-words), and a generic one-hot encoding of category labels
(here, the POS tags of Doc 0).

In [122]:
print("=== One-hot encoding of tokens (Doc 0) ===")
one_hot_tokens_df = one_hot_encode_tokens(processed_docs[0], vocab)
one_hot_tokens_df


=== One-hot encoding of tokens (Doc 0) ===


,agent,ai,also,anyth,apigenin,appreciate,archiv,awkward,behavior,behind,blender,bug,came,chang,cod,...,user,using,v,ve,video,vote,want,way,where,who,window,within,work,would,xai
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [123]:
print("=== One-hot encoding of POS tags (Doc 0) ===")
pos_tags_doc0 = [tag for _, tag in pos_tag(raw_tokens_per_doc[0])]
one_hot_pos_df = one_hot_encode_categories(pos_tags_doc0)
one_hot_pos_df


=== One-hot encoding of POS tags (Doc 0) ===


,ADJ,DET,NOUN,PREP
0,0,1,0,0
1,1,0,0,0
2,1,0,0,0
3,0,0,0,1
4,0,0,1,0


## 13. TF-IDF

In [124]:
tfidf_df = tf_idf(processed_docs, vocab)
tfidf_df.round(3)


,agent,ai,also,anyth,apigenin,appreciate,archiv,awkward,behavior,behind,blender,bug,came,chang,cod,...,user,using,v,ve,video,vote,want,way,where,who,window,within,work,would,xai
0,0.000,0.000,0.000,0.000,2.303,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,0.000,0.000,0.000,0.000,0.000,0.000,2.303,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.303,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.303
6,2.303,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.303,0.000,0.000,0.000,2.303,...,0.000,2.303,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,0.000,0.000,2.303,2.303,0.000,2.303,0.000,2.303,2.303,0.000,0.000,2.303,2.303,4.605,0.000,...,2.303,0.000,2.303,2.303,0.000,0.000,2.303,2.303,2.303,2.303,2.303,2.303,13.816,0.000,0.000
8,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,2.303,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.303,0.000
9,0.000,2.303,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,2.303,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


### Document Similarity (TF-IDF Cosine)

Pairwise cosine similarity between documents' TF-IDF vectors — lexical
overlap, not deep semantic meaning.

In [125]:
cosine_similarity_matrix(tfidf_df).round(3)

,doc_0,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9
doc_0,1.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000
doc_1,0.0,1.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000
doc_2,0.0,0.0,1.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000
doc_3,0.0,0.0,0.0,1.0,0.0,0.000,0.000,0.000,0.000,0.000
doc_4,0.0,0.0,0.0,0.0,1.0,0.000,0.000,0.000,0.000,0.000
doc_5,0.0,0.0,0.0,0.0,0.0,1.000,0.000,0.013,0.000,0.000
doc_6,0.0,0.0,0.0,0.0,0.0,0.000,1.000,0.013,0.000,0.000
doc_7,0.0,0.0,0.0,0.0,0.0,0.013,0.013,1.000,0.011,0.018
doc_8,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.011,1.000,0.000
doc_9,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.018,0.000,1.000


## 14. N-grams

Bigrams and trigrams for every document, built from the fully preprocessed
tokens.

In [126]:
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}")
    print("  Bigrams: ", n_grams(tokens, 2))
    print("  Trigrams:", n_grams(tokens, 3))


Doc 0
  Bigrams:  [('therapeutic', 'potential'), ('potential', 'apigenin')]
  Trigrams: [('therapeutic', 'potential', 'apigenin')]
Doc 1
  Bigrams:  [('interactive', 'physic')]
  Trigrams: []
Doc 2
  Bigrams:  [('economic', 'intelligence'), ('intelligence', 'frontier')]
  Trigrams: [('economic', 'intelligence', 'frontier')]
Doc 3
  Bigrams:  [('packslip', 'sign'), ('sign', 'release'), ('release', 'manifest'), ('manifest', 'archiv'), ('archiv', 'installer'), ('installer', 'executabl')]
  Trigrams: [('packslip', 'sign', 'release'), ('sign', 'release', 'manifest'), ('release', 'manifest', 'archiv'), ('manifest', 'archiv', 'installer'), ('archiv', 'installer', 'executabl')]
Doc 4
  Bigrams:  [('innovative', 'hp'), ('hp', 'computer'), ('computer', 'behind'), ('behind', 'hedge'), ('hedge', 'fund'), ('fund', 'pioneer')]
  Trigrams: [('innovative', 'hp', 'computer'), ('hp', 'computer', 'behind'), ('computer', 'behind', 'hedge'), ('behind', 'hedge', 'fund'), ('hedge', 'fund', 'pioneer')]
Doc 5
